# 🔬 Google Colab GPU Runner: Pretrained Foundation Models (P1–P5)
**Project:** Does Synthetic Data Really Help? Minority-Class Skin Lesion Benchmark  
**Backbones:** DINOv2 ViT-S/14 (`facebook/dinov2-small`) & BiomedCLIP (`microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224`)

**Protocol:** Extract embeddings ONCE per backbone on T4 GPU, cache to disk, and train classification heads across all 15 folds.


In [ ]:
# 1. Verify GPU Acceleration
import torch
assert torch.cuda.is_available(), 'GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU.'
print(f'Active GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Mount Google Drive & Setup Workspace
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/rare_class_benchmark'
os.makedirs(DRIVE_DIR, exist_ok=True)
if os.path.exists('/content/rare-class-benchmark'):
    %cd /content/rare-class-benchmark
elif os.path.exists('scripts/run_all.py'):
    pass
elif os.path.exists('/content/scripts/run_all.py'):
    %cd /content
else:
    !git clone https://github.com/dhina-research/rare-class-skin-lesion-benchmark.git /content/rare-class-benchmark 2>/dev/null || true
    if os.path.exists('/content/rare-class-benchmark'):
        %cd /content/rare-class-benchmark
!pip install -q transformers open_clip_torch timm scikit-learn pandas matplotlib seaborn pyyaml tqdm


In [ ]:
# 3. Extract and Cache Foundation Embeddings on GPU
from src.embed import extract_and_cache_embeddings
print('Extracting DINOv2 ViT-S/14 embeddings (384-d)...')
extract_and_cache_embeddings(backbone_name='facebook/dinov2-small')

print('Extracting BiomedCLIP embeddings (512-d)...')
extract_and_cache_embeddings(backbone_name='hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')


In [ ]:
# 4. Execute Pretrained Arms (P1–P5) Across All 15 Evaluations
# Runs in minutes on cached embeddings
!python scripts/run_all.py --arms P1 P2 P3 P4 P5 --seeds 42 1337 2024 --folds 0 1 2 3 4


In [ ]:
# 5. Statistical Aggregation: Foundation Models vs. CNN Baselines
from src.stats import aggregate_benchmark_runs
df_summary, df_runs = aggregate_benchmark_runs('results/logs/runs.csv', 'results/tables')
display(df_summary)

!cp -r results/logs/runs.csv {DRIVE_DIR}/ 2>/dev/null || true
!cp -r results/tables/ {DRIVE_DIR}/ 2>/dev/null || true
